In [ ]:
import json
from pathlib import Path

from fastapi import FastAPI, HTTPException
from fastapi.responses import FileResponse
from fastapi.staticfiles import StaticFiles

import os 
from dotenv import dotenv_values

import math
from datetime import datetime, timezone

import httpx
from fastapi import Query, HTTPException

import asyncio
import uvicorn

In [33]:
BASE_DIR = Path(r"C:\\Users\\Leeys\\Desktop\\yonsei\\마이 AI랩\\medinavi")

FRONTEND_DIR = BASE_DIR / "frontend"
DATA_PATH = BASE_DIR / "data" / "hospitals.json"

required_files = [FRONTEND_DIR / "index.html", FRONTEND_DIR / "map.js", DATA_PATH,]

for path in required_files:
    print(f"{path.name}: {'있음' if path.is_file() else '없음'}")

    if not path.is_file():
        raise FileNotFoundError(f"파일 위치를 확인하세요: {path}")

with DATA_PATH.open("r", encoding = "utf-8-sig") as file:
    hospital_data = json.load(file)

print("의료기관 수:", len(hospital_data["items"]))
print("첫 번째 기관:", hospital_data["items"][0]["dutyName"])

index.html: 있음
map.js: 있음
hospitals.json: 있음
의료기관 수: 23
첫 번째 기관: 강릉아산병원


In [50]:
app = FastAPI()

# frontend 폴더의 javascript 등의 파일 제공

app.mount("/static", StaticFiles(directory = str(FRONTEND_DIR)), name = "static",)

@app.get("/")
def home():
    return FileResponse(FRONTEND_DIR / "index.html")

@app.get("/api/hospitals")
def get_hospitals():
    try:
        with DATA_PATH.open("r", encoding = "utf-8-sig") as file:
            data = json.load(file)

    except FileNotFoundError:
        raise HTTPException(status_code = 500, detail = "의료기관 데이터 파일을 찾지 못했습니다.",)

    except json.JSONDecodeError:
        raise HTTPException(status_code = 500, detail = "의료기관 JSON 파일의 형식을 확인하세요.",)

    # 출처, 조회 시점, items를 포함한 기존 구조를 유지

    return data

In [51]:
settings = dotenv_values(BASE_DIR / ".env", encoding = "utf-8-sig",)

NAVER_CLIENT_ID = (settings.get("NAVER_MAPS_CLIENT_ID") or "").strip()
NAVER_CLIENT_SECRET = (settings.get("NAVER_MAPS_CLIENT_SECRET") or "").strip()

if not NAVER_CLIENT_ID or not NAVER_CLIENT_SECRET:
    raise ValueError(".env의 네이버 인증정보를 확인하세요.")

print("인증정보 불러오기 완료")

인증정보 불러오기 완료


In [67]:
DIRECTIONS_URL = "https://maps.apigw.ntruss.com/map-direction/v1/driving"

@app.get("/api/route")
async def get_route(hospital_id: str, start_lat: float = Query(..., ge = -90, le = 90), start_lon: float = Query(..., ge = -180, le = 180)):

    # 기존 병원 목록 기능에서 데이터를 읽음

    hospitals = get_hospitals()["items"]

    hospital = next((item for item in hospitals if item.get("hpid") == hospital_id), None,)

    if hospital is None:
        raise HTTPException(404, "의료기관을 찾지 못했습니다.")

    try:
        goal_lat = float(hospital["wgs84Lat"])
        goal_lon = float(hospital["wgs84Lon"])

    except (KeyError, TypeError, ValueError):
        raise HTTPException(status_code = 422, detail = "의료기관 좌표를 확인해야 합니다.") from None

    if not(math.isfinite(goal_lat) and math.isfinite(goal_lon) and -90 <= goal_lat <= 90 and -180 <= goal_lon <= 180):
        raise HTTPException(status_code = 422, detail = "의료기관 좌표가 유효한 범위를 벗어났습니다.",)

    # 성공한 진단 셀과 같은 방식으로 요청

    headers = {"x-ncp-apigw-api-key-id": NAVER_CLIENT_ID, "x-ncp-apigw-api-key": NAVER_CLIENT_SECRET,}

    # trafast: 실시간 빠른 길 

    params = {"start": f"{start_lon},{start_lat}", "goal": f"{goal_lon},{goal_lat}", "option": "trafast"}

    try:
        async with httpx.AsyncClient(timeout = 20) as client:
            response = await client.get(DIRECTIONS_URL, headers = headers, params = params,)

    except httpx.TimeoutException as error:
        raise HTTPException(status_code = 504, detail = f"길찾기 응답 시간이 초과({type(error).__name__})",) from None

    except httpx.RequestError as error:
        raise HTTPException(status_code = 502, detail = f"길찾기 연결 오류 ({type(error).__name__})",) from None

    try:
        data = response.json()

    except ValueError:
        raise HTTPException(status_code = 502, detail = "길찾기 응답을 JSON으로 읽지 못했습니다.",) from None

    if not isinstance(data, dict):
        raise HTTPException(status_code = 502, detail = "길찾기 응답 형식을 확인해 주세요.",)

    route_errors = {"1": "출발지와 목적지가 같은 지점으로 인식됐어요.", "2": "출발지 또는 병원 위치가 도로 주변에 있지 않아 경로를 찾지 못했어요.",
                    "3": "두 지점 사이의 자동차 경로를 찾지 못했어요.",}

    route_error = route_errors.get(str(data.get("code")))

    if response.status_code == 400 and route_error:
        raise HTTPException(status_code = 422, detail = route_error)

    if response.status_code != 200:
        raise HTTPException(status_code = 502, detail = f"네이버 길찾기에서 HTTP {response.status_code} 응답을 받았어요.",)

    if data.get("code") != 0:
        raise HTTPException(422, "경로를 찾지 못했어요. 출발지와 병원 위치를 확인해 주세요.")

    try:
        route = data["route"]["trafast"][0]
        summary = route["summary"]

        return {"hospital_id": hospital_id, "hospital_name": hospital.get("dutyName"), "distance_m": summary["distance"], "duration_ms": summary["duration"],
                "path": route["path"], "route_options": "trafast", "source": "NAVER Maps Direction 5", "queried_at": datetime.now(timezone.utc).isoformat(),}

    except (KeyError, IndexError, TypeError):
        raise HTTPException(status_code = 502, detail = "길찾기 결과에 필요한 정보가 없습니다.") from None

In [68]:
hospitals = get_hospitals()["items"]

test_start = hospitals[0]
test_goal = hospitals[1]

print("시험용 출발지:", test_start["dutyName"])
print("목적지:", test_goal["dutyName"])

try:
    result = await get_route(
        hospital_id=test_goal["hpid"],
        start_lat=float(test_start["wgs84Lat"]),
        start_lon=float(test_start["wgs84Lon"]),
    )

    print("이동거리:", round(result["distance_m"] / 1000, 1), "km")
    print("예상 이동시간:", math.ceil(result["duration_ms"] / 60000), "분")
    print("경로 좌표 수:", len(result["path"]))

except HTTPException as error:
    print("조회 실패:", error.detail)

시험용 출발지: 강릉아산병원
목적지: 강원대학교병원
이동거리: 148.7 km
예상 이동시간: 100 분
경로 좌표 수: 1704


In [ ]:
# 같은 노트북에서 서버를 중복 실행하지 않도록 확인 
# 핸드폰으로 화면, 지도, 병원 목록 확인용 서버

if "server_task" in globals() and not server_task.done():
    print("서버가 이미 실행중입니다.")

else:
    config = uvicorn.Config(app = app, host = "0.0.0.0", port = 8000, reload = False, log_level = "info",)

    server = uvicorn.Server(config)

    # 서버가 실행되는 동안에도 노트북의 다른 셀을 사용할 수 있음

    server_task = asyncio.create_task(server.serve())

    await asyncio.sleep(1)

    if server.started:
        print("서버 실행 완료")
        print("메디네비: http://127.0.0.1:8000/")
        print("병원 목록: http://127.0.0.1:8000/api/hospitals")
    else:
        print("위의 서버 로그에서 실행 상태를 확인하세요.")

INFO:     Started server process [10656]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


서버 실행 완료
메디네비: http://127.0.0.1:8000/
병원 목록: http://127.0.0.1:8000/api/hospitals


INFO:     127.0.0.1:61449 - "GET / HTTP/1.1" 200 OK
INFO:     127.0.0.1:61449 - "GET /static/map.js?v=5 HTTP/1.1" 200 OK
INFO:     127.0.0.1:61449 - "GET /api/hospitals HTTP/1.1" 200 OK
INFO:     127.0.0.1:52977 - "GET /api/route?hospital_id=A2200015&start_lat=37.277186&start_lon=127.910501 HTTP/1.1" 200 OK
INFO:     115.86.114.89:0 - "GET / HTTP/1.1" 200 OK
INFO:     115.86.114.89:0 - "GET /static/map.js?v=5 HTTP/1.1" 200 OK
INFO:     115.86.114.89:0 - "GET /api/hospitals HTTP/1.1" 200 OK
INFO:     115.86.114.89:0 - "GET /favicon.ico HTTP/1.1" 404 Not Found
INFO:     115.86.114.89:0 - "GET / HTTP/1.1" 200 OK
INFO:     115.86.114.89:0 - "GET /static/map.js?v=5 HTTP/1.1" 200 OK
INFO:     115.86.114.89:0 - "GET /api/hospitals HTTP/1.1" 200 OK


In [ ]:
# 같은 노트북에서 서버를 중복 실행하지 않도록 확인

if "server_task" in globals() and not server_task.done():
    print("서버가 이미 실행중입니다.")

else:
    config = uvicorn.Config(app = app, host = "127.0.0.1", port = 8000, reload = False, log_level = "info",)

    server = uvicorn.Server(config)

    # 서버가 실행되는 동안에도 노트북의 다른 셀을 사용할 수 있음

    server_task = asyncio.create_task(server.serve())

    await asyncio.sleep(1)

    if server.started:
        print("서버 실행 완료")
        print("메디네비: http://127.0.0.1:8000/")
        print("병원 목록: http://127.0.0.1:8000/api/hospitals")
    else:
        print("위의 서버 로그에서 실행 상태를 확인하세요.")

INFO:     Started server process [10656]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:8000 (Press CTRL+C to quit)


서버 실행 완료
메디네비: http://127.0.0.1:8000/
병원 목록: http://127.0.0.1:8000/api/hospitals


INFO:     115.86.19.26:0 - "GET / HTTP/1.1" 200 OK
INFO:     115.86.19.26:0 - "GET /static/map.js HTTP/1.1" 200 OK
INFO:     115.86.19.26:0 - "GET /api/hospitals HTTP/1.1" 200 OK
INFO:     115.86.19.26:0 - "GET /favicon.ico HTTP/1.1" 404 Not Found


INFO:     Shutting down
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.
INFO:     Finished server process [10656]


In [71]:
if "server_task" in globals() and not server_task.done():
    server.should_exit = True
    await server_task
    print("서버를 중지했습니다.")
else:
    print("실행 중인 서버가 없습니다.")

서버를 중지했습니다.
